# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`This notebook demonstrates how to load and explore the FAIR² mass spectrometry dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described using a [Croissant schema](https://mlcommons.org/croissant/) available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install `mlcroissant` if necessary
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset schema and metadata
dataset = mlc.Dataset(croissant_url)
# 'to_json()' provides the full metadata as a Python dict
metadata = dataset.metadata.to_json()

print(f"{metadata['name']}:")
print(metadata['description'])

## 2. Data Overview
Croissant datasets are organized into one or more `recordSet`s, each with fields and columns described by unique `@id` identifiers. 

Let's inspect the record sets, their `@id`, and their fields.

In [ ]:
# List available record sets and their fields, by @id
if hasattr(dataset, 'record_sets'):
    print("Record sets in this dataset:")
    for rset in dataset.record_sets:
        print(f"- Record set name: {getattr(rset, 'name', getattr(rset, '@id', 'N/A'))}")
        print(f"  @id: {getattr(rset, '@id', 'N/A')}")
        if hasattr(rset, 'fields') and rset.fields:
            print("  Fields and their @ids:")
            for field in rset.fields:
                print(f"    - {getattr(field, 'name', getattr(field, '@id', 'N/A'))} (@id: {getattr(field, '@id', 'N/A')})")
        print("")
else:
    print('No record sets found in this dataset. Dataset may define record sets at a different metadata key.')

We can also list all available record set `@id`s to use in our data extraction steps.

In [ ]:
# Retrieve record set @ids as a list (required for later extraction)
record_sets = []
if hasattr(dataset, 'record_sets'):
    for rset in dataset.record_sets:
        record_sets.append(getattr(rset, '@id', None))
print("RecordSet @ids:", record_sets)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 
Always reference record sets and fields by their exact `@id` from the overview.

In [ ]:
# We'll extract data from each record set and load it into Pandas DataFrames.
dataframes = {}

for rset_id in record_sets:
    print(f'Loading records for record set @id: {rset_id}')
    try:
        records = list(dataset.records(record_set=rset_id))
        # Flatten if records are nested dicts
        df = pd.json_normalize(records)
        dataframes[rset_id] = df
        print(f"  -> {len(df)} records loaded. Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"  !! Could not load records: {e}")

# For demonstration, display columns and first rows for the first record set:
if record_sets:
    first_rset = record_sets[0]
    print(f'Columns for {first_rset}:')
    print(dataframes[first_rset].columns.tolist())
    dataframes[first_rset].head()

## 4. Exploratory Data Analysis (EDA)
Let's apply standard data processing steps on the main record set.

**Choose a numeric field by its `@id` for filtering, normalization, and grouping.**

In [ ]:
# Identify a numeric field for EDA. Replace with valid field @id from overview if necessary.
main_rset_id = record_sets[0] if len(record_sets) > 0 else None
main_df = dataframes[main_rset_id].copy() if main_rset_id else None

numeric_field = None
# Try to auto-detect a likely numeric field by name
if main_df is not None:
    for col in main_df.columns:
        # Examples: 'MW', 'molecular_weight', 'abundance', etc.
        if any(x in col.lower() for x in ['abundance', 'mw', 'weight', 'count', 'coverage', 'number']) and pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field = col
            break
if numeric_field is None and main_df is not None:
    numeric_candidates = main_df.select_dtypes(include='number').columns.tolist()
    numeric_field = numeric_candidates[0] if numeric_candidates else None

print(f"Numeric field selected: {numeric_field}")

# Example threshold (may adjust as needed)
threshold = 10
if main_df is not None and numeric_field is not None:
    filtered_df = main_df[main_df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by protein or gene column, if present
    group_field = None
    for col in main_df.columns:
        if any(x in col.lower() for x in ['protein', 'gene', 'group', 'category', 'description']):
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())
else:
    print('No numeric field found or dataframe is empty.')

## 5. Visualization
Visualize numeric field distributions and relationships.

Examples: histogram of abundance, scatterplot of molecular weight vs. abundance, etc.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_df is not None and numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field], kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If two likely numeric columns, do a scatterplot
    numeric_candidates = main_df.select_dtypes(include='number').columns.tolist()
    if len(numeric_candidates) >= 2:
        plt.figure(figsize=(6,6))
        sns.scatterplot(data=main_df, x=numeric_candidates[0], y=numeric_candidates[1])
        plt.title(f"Scatterplot: {numeric_candidates[0]} vs {numeric_candidates[1]}")
        plt.xlabel(numeric_candidates[0])
        plt.ylabel(numeric_candidates[1])
        plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load and explore the FAIR² extracellular vesicle proteomics dataset described by a Croissant schema.

- Dataset record sets, fields, and their unique `@id`s were listed for robust programmatic access.
- Data from each record set was loaded and shown in tabular format.
- Standard exploratory data analysis and visualizations were applied using Python.

You can now extend this notebook to apply your own domain-specific analysis—using the exact `@id`s for all referencing in the dataset schema.